In [11]:
# %% [markdown]
# # LLM-Guided Counterfactual Intervention for Comorbid Chronic Disease Risk
# **Framework**: XGBoost (Stage 1) → GPT-4o-mini Guardrail Agent (Stage 2) → DiCE (Stage 3)
# **Data**: KNHANES 2020–2024

# %% [markdown]
# ## 0. Library Import & Settings

# %%
import json
import joblib
import numpy as np
import optuna
import pandas as pd
import pyreadstat
import xgboost as xgb
import openai
import dice_ml

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.max_colwidth', None)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# OpenAI API key
OPENAI_API_KEY = "YOUR_API_KEY_HERE"   # ← Enter your API key here
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# %% [markdown]
# ## 1. KNHANES 2020–2024 Data Merge

# %%
df20, meta20 = pyreadstat.read_sas7bdat('hn20_all.sas7bdat')
df21, meta21 = pyreadstat.read_sas7bdat('hn21_all.sas7bdat')
df22, meta22 = pyreadstat.read_sas7bdat('hn22_all.sas7bdat')
df23, meta23 = pyreadstat.read_sas7bdat('hn23_all.sas7bdat')
df24, meta24 = pyreadstat.read_sas7bdat('hn24_all.sas7bdat')

# %% [markdown]
# ### 1-1. Variable Selection

# %%
KEY_COLS    = ['ID', 'year', 'sex']
CAT_COLS    = [
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3', 'BD1_11', 'BD2_1', 'BS3_1',
    'BE3_71', 'BE3_75', 'BE3_81', 'BE3_91', 'pa_aerobic', 'L_BR_FQ',
    'BP1', 'mh_stress', 'incm', 'ho_incm', 'edu', 'BH1',
]
NUM_COLS    = [
    'HE_BMI', 'HE_wc', 'HE_wt',
    'N_EN', 'N_CHO', 'N_SUGAR', 'N_NA', 'N_FAT',
    'N_SFA', 'N_TDF', 'N_K', 'N_PROT',
]
TARGET_COLS = ['HE_DM_HbA1c', 'HE_HP']
ALL_VARS    = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

# %% [markdown]
# ### 1-2. Filter & Merge

# %%
df_total = pd.concat(
    [d[ALL_VARS].copy() for d in [df20, df21, df22, df23, df24]], axis=0
).reset_index(drop=True)

# %% [markdown]
# ### 1-3. Target Variable Encoding

# %%
# HE_DM_HbA1c: 1=Normal→0, 3=Diabetes→1
df_total['HE_DM_HbA1c'] = df_total['HE_DM_HbA1c'].map({1: 0, 3: 1}).fillna(-999)
# HE_HP: 1=Normal→0, 4=Hypertension→1
df_total['HE_HP']        = df_total['HE_HP'].map({1: 0, 4: 1}).fillna(-999)

for col in ['HE_DM_HbA1c', 'HE_HP']:
    print(f"\n{col} value counts:\n", df_total[col].value_counts())

# %% [markdown]
# ### 1-4. Integrated 4-Class Risk Label

# %%
# Class 0: Normal | Class 1: HTN-only | Class 2: DM-only | Class 3: Comorbid
conditions = [
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 1),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 1),
]
df_total['integrated_target'] = np.select(conditions, [0, 1, 2, 3], default=np.nan)
df_total = df_total.dropna().reset_index(drop=True)
df_total['integrated_target'] = df_total['integrated_target'].astype(int)

print("Risk class distribution:")
print(df_total['integrated_target'].value_counts().sort_index())

# %% [markdown]
# ### 1-5. Rename Columns: SAS codes → English labels

# %%
# English column mapping (SAS variable code → English label)
ENGLISH_LABEL_DICT = {
    'ID':           'ID',
    'year':         'SurveyYear',
    'sex':          'Sex',
    'HE_DM_HbA1c':  'Diabetes',
    'HE_HP':        'Hypertension',
    'integrated_target': 'RiskClass',
    # Categorical
    'HE_obe':   'ObesityStatus',
    'BO1_1':    'WeightChangeStatus',
    'BO1_2':    'WeightLossAmount',
    'BO1_3':    'WeightGainAmount',
    'BD1_11':   'DrinkingFrequency',
    'BD2_1':    'DrinkingAmount',
    'BS3_1':    'SmokingStatus',
    'BE3_71':   'VigorousActivity_Work',
    'BE3_75':   'VigorousActivity_Leisure',
    'BE3_81':   'ModerateActivity_Work',
    'BE3_91':   'WalkingActivity',
    'pa_aerobic': 'AerobicActivityRate',
    'L_BR_FQ':  'BreakfastFrequency',
    'BP1':      'StressLevel',
    'mh_stress':'StressAwarenessRate',
    'incm':     'PersonalIncomeQuartile',
    'ho_incm':  'HouseholdIncomeQuartile',
    'edu':      'EducationLevel',
    'BH1':      'HealthScreeningStatus',
    # Numerical
    'HE_BMI':   'BMI',
    'HE_wc':    'WaistCirc',
    'HE_wt':    'Weight',
    'N_EN':     'Energy_kcal',
    'N_CHO':    'Carb_g',
    'N_SUGAR':  'Sugar_g',
    'N_NA':     'Sodium_mg',
    'N_FAT':    'Fat_g',
    'N_SFA':    'SaturatedFat_g',
    'N_TDF':    'Fiber_g',
    'N_K':      'Potassium_mg',
    'N_PROT':   'Protein_g',
}

df_total.rename(columns=ENGLISH_LABEL_DICT, inplace=True)
df_total.to_csv('./knhanes_comorbid_2020_2024.csv', index=False, encoding='utf-8-sig')
print(">>> Preprocessing complete. CSV saved.")
df_total.head()

# %% [markdown]
# ## 2. Feature Definition & Numeric Dataset

# %%
NUM_FEATURES = [
    'BMI', 'WaistCirc', 'Weight',
    'Energy_kcal', 'Carb_g', 'Sugar_g', 'Sodium_mg',
    'Fat_g', 'SaturatedFat_g', 'Fiber_g', 'Potassium_mg', 'Protein_g',
]

CAT_FEATURES = [
    'ObesityStatus', 'WeightChangeStatus', 'WeightLossAmount', 'WeightGainAmount',
    'DrinkingFrequency', 'DrinkingAmount', 'SmokingStatus',
    'VigorousActivity_Work', 'VigorousActivity_Leisure',
    'ModerateActivity_Work', 'WalkingActivity', 'AerobicActivityRate',
    'BreakfastFrequency', 'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]

X_FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET_COL  = 'RiskClass'

# Variables excluded from DiCE intervention search:
# These are either non-modifiable (stress perception, income, education)
# or administrative (health screening status).
# DiCE will hold them fixed at the patient's current value.
FIXED_FEATURES = [
    'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]
VARY_FEATURES = [f for f in X_FEATURES if f not in FIXED_FEATURES]

# Build all-numeric dataset for XGBoost & DiCE
required_cols = X_FEATURES + [TARGET_COL, 'Sex']
df_final = df_total[required_cols].copy()
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
df_final = df_final.astype(float)

print(f"df_final shape : {df_final.shape}")
print(f"Any string column: {(df_final.dtypes == 'object').any()}")

# %% [markdown]
# ## 3. Stage 1 — XGBoost Gender-Stratified Training (Optuna)

# %%
def train_model(gender_code: float, gender_name: str):
    print(f"\n{'='*20} {gender_name} Model Training {'='*20}")

    df_g = df_final[df_final['Sex'] == gender_code].copy()
    X = df_g[X_FEATURES]
    y = df_g[TARGET_COL].astype(int)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    sw = compute_sample_weight('balanced', y=y_train)

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'objective':        'multi:softprob',
            'num_class':        4,
            'tree_method':      'hist',
            'random_state':     42,
        }
        mdl = xgb.XGBClassifier(**params)
        mdl.fit(X_train, y_train, sample_weight=sw)
        return f1_score(y_val, mdl.predict(X_val), average='weighted')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)

    best_model = xgb.XGBClassifier(
        **study.best_params,
        objective='multi:softprob',
        num_class=4,
        tree_method='hist',
        random_state=42,
    )
    best_model.fit(X_train, y_train, sample_weight=sw)

    print(f"\n[{gender_name}] Classification Report:")
    print(classification_report(y_val, best_model.predict(X_val)))

    return best_model, study.best_params

# Train male (Sex=1) and female (Sex=2)
model_male,   params_male   = train_model(1.0, "Male")
model_female, params_female = train_model(2.0, "Female")

# Save models
agent_config = {
    'X_features':    X_FEATURES,
    'num_features':  NUM_FEATURES,
    'cat_features':  CAT_FEATURES,
    'target_col':    TARGET_COL,
    'params_male':   params_male,
    'params_female': params_female,
}
joblib.dump(model_male,    'model_male.pkl')
joblib.dump(model_female,  'model_female.pkl')
joblib.dump(agent_config,  'agent_config.pkl')
joblib.dump(df_final,      'df_final.pkl')
print(">>> Models and config saved.")

# %% [markdown]
# ## 4. Stage 2 — GPT-4o-mini Clinical Guardrail Agent

# %%
def get_clinical_guardrails(patient_data: pd.DataFrame,
                             gender_name: str,
                             allowed_features: list,
                             metadata: dict) -> dict:
    """
    Calls GPT-4o-mini to derive patient-specific permitted_range constraints.
    Returns a dict with keys: 'reasoning', 'guardrail_ranges'.

    Key fix: BMI upper bound is set to current value * 1.0 (allow reduction),
    and WaistCirc is coupled to Weight/BMI direction to prevent
    physically inconsistent combinations (e.g. WaistCirc drops while
    Weight/BMI are unchanged).
    """
    patient_info = patient_data.to_dict(orient='records')[0]

    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurance company.
You must respond ONLY in valid JSON using the exact variable names listed below.

[Patient Information ({gender_name})]
{json.dumps(patient_info, ensure_ascii=False, indent=1)}

[Available Variable Names]
{", ".join(allowed_features)}

[Categorical Variable Valid Values (use only these values)]
{json.dumps(metadata, ensure_ascii=False, indent=1)}

[Guardrail Principles — apply ALL of the following strictly]

1. CONFLICT PREVENTION:
   - Cap Carb_g at [0, current Carb_g] and Sugar_g at [0, current Sugar_g]
     whenever Sodium_mg is reduced below its current value.
   - This prevents compensatory carbohydrate intake that worsens glycaemic control.

2. PHYSICAL CONSISTENCY (critical — violations found in prior run):
   - BMI range: [current_BMI * 0.85, current_BMI].
     The upper bound must NOT exceed the current value; reduction is the goal.
   - WaistCirc range: [current_WaistCirc * 0.85, current_WaistCirc].
     WaistCirc MUST decrease together with BMI/Weight; it cannot decrease
     independently while BMI and Weight remain unchanged.
   - Weight range: [current_Weight * 0.85, current_Weight].
   - All three (BMI, WaistCirc, Weight) must move in the same direction.
     If only dietary composition changes (no weight loss pathway), set all three
     to [current * 0.97, current] to keep them nearly fixed but consistent.

3. ENERGY FLOOR:
   - Energy_kcal lower bound = max(current_Energy_kcal * 0.70, 500).
   - Upper bound = current_Energy_kcal (do not allow caloric increase).

4. SAFETY: All lower bounds must be >= 0.

5. DIVERSITY HINT:
   - Set Fat_g range to allow both reduction and mild increase
     so that DiCE can generate structurally distinct pathways
     (weight-loss-led vs. dietary-composition-led).

[Response Format — valid JSON only, no extra text]
{{
    "reasoning": "Detailed clinical rationale explaining each constraint choice",
    "guardrail_ranges": {{ "ExactVariableName": [min, max] }}
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",
             "content": "You are an expert who responds strictly in valid JSON, "
                        "fully adhering to clinical guidelines and the provided "
                        "data schema. Never allow WaistCirc to decrease while "
                        "BMI and Weight remain unchanged."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# %% [markdown]
# ## 5. Stage 3 — DiCE Counterfactual Generation with Stepwise Fallback

# %%
def run_dice_pipeline(model, df_input: pd.DataFrame,
                      gender_code: float, gender_name: str):
    """
    Full Stage 2+3 pipeline:
      1. Sample a Class-3 patient
      2. Generate LLM guardrails
      3. Apply physical correction layer
      4. Run DiCE with stepwise fallback (Class 0 → 1 → 2)
    Returns the DiCE result object or None.
    """
    print(f"\n{'='*25} {gender_name} Pipeline {'='*25}")

    df_stable = df_input.copy().astype(float)

    # Sample a Class-3 patient that the model also predicts as Class 3
    # (confirmed comorbid case). Additionally require BMI >= 23 to avoid
    # edge cases where the patient is already underweight/borderline,
    # which causes the LLM to generate physiologically implausible guardrails.
    class3_pool = df_stable[
        (df_stable['Sex'] == gender_code) &
        (df_stable[TARGET_COL] == 3.0) &
        (df_stable['BMI'] >= 23.0)
    ].copy()

    preds     = model.predict(class3_pool[X_FEATURES])
    confirmed = class3_pool[preds == 3]

    if len(confirmed) == 0:
        print("⚠️  No model-confirmed Class 3 patients (BMI≥23). Falling back to all Class 3.")
        confirmed = class3_pool

    sample = confirmed.sample(1, random_state=42)
    query  = sample[X_FEATURES]
    print(f"Sampled patient — true class: {int(sample[TARGET_COL].values[0])}, "
          f"predicted class: {int(model.predict(query)[0])}, "
          f"BMI: {query['BMI'].values[0]:.2f}")

    # 2. Categorical valid-value metadata
    feature_metadata = {
        col: sorted(df_stable[col].unique().tolist())
        for col in CAT_FEATURES if col in df_stable.columns
    }

    # 3. LLM guardrails
    print("Calling GPT-4o-mini guardrail agent...")
    agent_plan = get_clinical_guardrails(query, gender_name, X_FEATURES, feature_metadata)
    print("Reasoning:", agent_plan.get('reasoning', '')[:300], "...")

    # 4. Physical correction layer
    # All rules below unconditionally override the LLM output.
    cur_bmi    = float(query.iloc[0]['BMI'])
    cur_waist  = float(query.iloc[0]['WaistCirc'])
    cur_weight = float(query.iloc[0]['Weight'])
    cur_energy = float(query.iloc[0]['Energy_kcal'])
    cur_sodium = float(query.iloc[0]['Sodium_mg'])
    cur_carb   = float(query.iloc[0]['Carb_g'])
    cur_sugar  = float(query.iloc[0]['Sugar_g'])
    cur_prot   = float(query.iloc[0]['Protein_g'])
    cur_pot    = float(query.iloc[0]['Potassium_mg'])

    # ── Step 1: Start from LLM ranges, apply absolute floor of 0 ─────────────
    valid_guardrails = {}
    for feat in X_FEATURES:
        cur = float(query.iloc[0][feat])
        if feat in agent_plan['guardrail_ranges']:
            lo = max(float(agent_plan['guardrail_ranges'][feat][0]), 0.0)
            hi = max(float(agent_plan['guardrail_ranges'][feat][1]), lo)
        else:
            lo, hi = max(cur * 0.7, 0.0), cur * 1.3
        valid_guardrails[feat] = [lo, hi]

    # ── Rule A: Anthropometric consistency (hard override) ───────────────────
    # BMI, WaistCirc, Weight move together in the same direction.
    # Range: [85%, 100%] of current value for all three.
    # Additionally enforce: if BMI changes, Weight MUST also change
    # (they are physically coupled via height, which is constant).
    for feat, cur in [('BMI', cur_bmi), ('WaistCirc', cur_waist), ('Weight', cur_weight)]:
        valid_guardrails[feat] = [round(cur * 0.85, 4), round(cur * 1.00, 4)]

    # Tighten WaistCirc floor so its % reduction ≤ Weight % reduction
    weight_floor_ratio = valid_guardrails['Weight'][0] / cur_weight   # = 0.85
    valid_guardrails['WaistCirc'][0] = max(
        valid_guardrails['WaistCirc'][0],
        round(cur_waist * weight_floor_ratio, 4)
    )
    # BMI floor ratio tied to Weight floor ratio (same person, constant height)
    valid_guardrails['BMI'][0] = max(
        valid_guardrails['BMI'][0],
        round(cur_bmi * weight_floor_ratio, 4)
    )

    # ── Rule B: Energy floor (70 %) and ceiling (current) ────────────────────
    # Floor: 70% of current intake (prevents muscle catabolism / deficiency).
    # Ceiling: current intake (no caloric increase permitted).
    valid_guardrails['Energy_kcal'] = [
        round(max(cur_energy * 0.70, 500.0), 4),
        round(cur_energy, 4)
    ]

    # ── Rule C: Sodium – prevent hyponatraemia ────────────────────────────────
    # Floor: max(60% of current, 800 mg/day) per WHO safe intake guidance.
    # Ceiling: 90% of current (force at least 10% reduction for HTN benefit).
    sodium_floor   = round(max(cur_sodium * 0.60, 800.0), 4)
    sodium_ceiling = round(cur_sodium * 0.90, 4)
    valid_guardrails['Sodium_mg'] = [sodium_floor, sodium_ceiling]

    # ── Rule D: Conflict prevention (Sodium↓ → Carb/Sugar cap) ───────────────
    # Always cap Carb and Sugar at current values to block compensatory intake.
    valid_guardrails['Carb_g'][1]  = round(cur_carb,  4)
    valid_guardrails['Sugar_g'][1] = round(cur_sugar, 4)

    # ── Rule E: Nutritional minimums (prevent deficiency pathways) ────────────
    # Protein  : floor at 60% of current or 30 g (sarcopenia prevention)
    # Potassium: floor at 50% of current or 500 mg (cardiac safety)
    # Carb     : floor at 20% of current or 30 g (brain glucose minimum)
    # Fiber    : floor at 30% of current or 5 g (gut health minimum)
    cur_fiber = float(query.iloc[0]['Fiber_g'])
    valid_guardrails['Protein_g'][0]    = round(max(cur_prot  * 0.60, 30.0), 4)
    valid_guardrails['Potassium_mg'][0] = round(max(cur_pot   * 0.50, 500.0), 4)
    valid_guardrails['Carb_g'][0]       = round(max(cur_carb  * 0.20, 30.0), 4)
    valid_guardrails['Fiber_g'][0]      = round(max(cur_fiber * 0.30, 5.0),  4)

    # Final sanity check: ensure lo <= hi for every feature
    for feat in X_FEATURES:
        lo, hi = valid_guardrails[feat]
        if lo > hi:
            cur = float(query.iloc[0][feat])
            valid_guardrails[feat] = [round(cur * 0.85, 4), round(cur * 1.0, 4)]

    print("\nFinal guardrail ranges (sample):")
    for k, v in list(valid_guardrails.items())[:6]:
        print(f"  {k}: {v}")

    # 5. DiCE setup
    # NOTE: DiCE's genetic method does not reliably support categorical feature
    # declarations — declaring categories causes it to collapse the search space
    # and find zero counterfactuals. We keep all features as continuous and
    # instead enforce integer validity through the permitted_range guardrails
    # (categorical variables already have narrow integer-aligned ranges from
    # the feature_metadata lookup above, which limits DiCE to valid values).
    d   = dice_ml.Data(
            dataframe=df_stable[X_FEATURES + [TARGET_COL]],
            continuous_features=X_FEATURES,
            outcome_name=TARGET_COL,
          )
    m   = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="genetic")

    # 6. Stepwise fallback: Class 0 → 1 → 2
    fallback_targets = {0: "Normal", 1: "HTN-only", 2: "DM-only"}
    for target_class, target_name in fallback_targets.items():
        print(f"\n▶ Searching for [{target_name}] (Class {target_class})...")
        for attempt in range(15):   # increased from 10 → 15 for narrow search spaces
            try:
                cf = exp.generate_counterfactuals(
                    query,
                    total_CFs=4,
                    desired_class=target_class,
                    features_to_vary=VARY_FEATURES,   # exclude non-modifiable variables
                    permitted_range=valid_guardrails,
                    proximity_weight=0.2,
                    sparsity_weight=0.1,
                )
                if cf and len(cf.cf_examples_list[0].final_cfs_df) > 0:
                    cf_df = cf.cf_examples_list[0].final_cfs_df.copy()

                    # ── Post-process 1: Round categorical columns to nearest valid integer
                    vary_cat = [f for f in CAT_FEATURES if f not in FIXED_FEATURES]
                    for col in vary_cat:
                        if col in cf_df.columns:
                            valid_vals = sorted(df_stable[col].dropna().unique().tolist())
                            cf_df[col] = cf_df[col].apply(
                                lambda v: min(valid_vals, key=lambda x: abs(x - v))
                            )

                    # ── Post-process 2: Enforce anthropometric coupling per CF row
                    # Problem: using a single min-ratio collapses diversity across CFs.
                    # Fix: per row, find which variable changed most (largest reduction),
                    # then scale the others to match that ratio — preserving row-level
                    # diversity while enforcing physical consistency within each row.
                    orig_bmi   = float(query.iloc[0]['BMI'])
                    orig_waist = float(query.iloc[0]['WaistCirc'])
                    orig_wt    = float(query.iloc[0]['Weight'])

                    def enforce_anthro_coupling(row):
                        b  = row['BMI']
                        w  = row['WaistCirc']
                        wt = row['Weight']
                        changed = (
                            abs(b  - orig_bmi)   > 0.01 or
                            abs(w  - orig_waist) > 0.01 or
                            abs(wt - orig_wt)    > 0.01
                        )
                        if not changed:
                            return row
                        # Per-variable reduction ratio (lower = larger reduction)
                        r_bmi   = b  / orig_bmi
                        r_waist = w  / orig_waist
                        r_wt    = wt / orig_wt
                        # The variable that changed the most sets the target ratio
                        r_target = min(r_bmi, r_waist, r_wt)
                        # Clamp to [0.85, 1.0] to stay within guardrail bounds
                        r_target = max(0.85, min(r_target, 1.0))
                        # Apply target ratio only to variables that did NOT change
                        # (or changed less than the leader) — preserves the leader's value
                        if r_bmi   > r_target:
                            row['BMI']       = round(orig_bmi   * r_target, 4)
                        if r_waist > r_target:
                            row['WaistCirc'] = round(orig_waist * r_target, 4)
                        if r_wt    > r_target:
                            row['Weight']    = round(orig_wt    * r_target, 4)
                        return row

                    cf_df = cf_df.apply(enforce_anthro_coupling, axis=1)
                    cf.cf_examples_list[0].final_cfs_df = cf_df

                    print(f"✅ Found {target_name} pathways on attempt {attempt + 1}!")
                    cf.visualize_as_dataframe(show_only_changes=True)
                    return cf
            except Exception:
                continue
        print(f"⚠️  [{target_name}] unreachable. Falling back to next target.")

    print("✖ All targets exhausted. No valid counterfactuals found.")
    return None

# %% [markdown]
# ## 6. Run Full Pipeline

# %%
cf_male   = run_dice_pipeline(model_male,   df_final, 1.0, "Male")
cf_female = run_dice_pipeline(model_female, df_final, 2.0, "Female")

# %% [markdown]
# ## 7. Pre-Guardrail Baseline: Unconstrained DiCE (Table 6 comparison)

# %%
def run_dice_no_guardrail(model, df_input: pd.DataFrame,
                           gender_code: float, gender_name: str):
    """
    Runs DiCE WITHOUT any guardrail constraints on the same patient sample.
    Used to produce the 'No guardrail (pure DiCE)' column in Table 6,
    demonstrating numerical hallucination and intervention conflicts.
    """
    print(f"\n{'='*20} {gender_name} — No-Guardrail Baseline {'='*20}")

    df_stable = df_input.copy().astype(float)
    class3_pool = df_stable[
        (df_stable['Sex'] == gender_code) &
        (df_stable[TARGET_COL] == 3.0) &
        (df_stable['BMI'] >= 23.0)
    ].copy()
    preds     = model.predict(class3_pool[X_FEATURES])
    confirmed = class3_pool[preds == 3]
    if len(confirmed) == 0:
        confirmed = class3_pool
    sample = confirmed.sample(1, random_state=42)
    query  = sample[X_FEATURES]

    d   = dice_ml.Data(
            dataframe=df_stable[X_FEATURES + [TARGET_COL]],
            continuous_features=X_FEATURES,
            outcome_name=TARGET_COL,
          )
    m   = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="genetic")

    try:
        cf = exp.generate_counterfactuals(
            query,
            total_CFs=4,
            desired_class=0,
            features_to_vary=VARY_FEATURES,
            proximity_weight=0.2,
            sparsity_weight=0.1,
        )
        if cf and len(cf.cf_examples_list[0].final_cfs_df) > 0:
            print("No-guardrail DiCE result (check for hallucinations):")
            cf.visualize_as_dataframe(show_only_changes=True)
            return cf
    except Exception as e:
        print(f"No-guardrail search failed: {e}")
    return None

cf_male_baseline   = run_dice_no_guardrail(model_male,   df_final, 1.0, "Male")
cf_female_baseline = run_dice_no_guardrail(model_female, df_final, 2.0, "Female")

# %% [markdown]
# ## 8. Export Results to JSON

# %%
def save_results_to_json(male_res, female_res,
                          male_base=None, female_base=None,
                          filename="experiment_results.json"):
    export = {
        "metadata": {
            "description": "LLM-Guided Counterfactual Intervention for Comorbid Risk",
            "model":        "XGBoost + GPT-4o-mini + DiCE",
            "data":         "KNHANES 2020-2024",
            "total_cfs_per_patient": 4,
        },
        "results": [],
        "baseline_no_guardrail": [],   # Table 6 comparison data
    }

    for res, label in zip([male_res, female_res], ["Male", "Female"]):
        if res is not None:
            entry = {
                "gender":          label,
                "status":          "Success",
                "original_data":   res.cf_examples_list[0].test_instance_df
                                      .to_dict(orient='records')[0],
                "counterfactuals": res.cf_examples_list[0].final_cfs_df
                                      .to_dict(orient='records'),
            }
        else:
            entry = {
                "gender":  label,
                "status":  "Failed",
                "message": "No counterfactuals found within guardrail constraints.",
            }
        export["results"].append(entry)

    for res, label in zip([male_base, female_base], ["Male", "Female"]):
        if res is not None:
            entry = {
                "gender":          label,
                "original_data":   res.cf_examples_list[0].test_instance_df
                                      .to_dict(orient='records')[0],
                "counterfactuals": res.cf_examples_list[0].final_cfs_df
                                      .to_dict(orient='records'),
            }
            export["baseline_no_guardrail"].append(entry)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export, f, ensure_ascii=False, indent=4)
    print(f"✅ Results saved to {filename}")

    for res, label in zip([male_res, female_res], ["Male", "Female"]):
        if res is not None:
            entry = {
                "gender":           label,
                "status":           "Success",
                "original_data":    res.cf_examples_list[0].test_instance_df
                                       .to_dict(orient='records')[0],
                "counterfactuals":  res.cf_examples_list[0].final_cfs_df
                                       .to_dict(orient='records'),
            }
        else:
            entry = {
                "gender":  label,
                "status":  "Failed",
                "message": "No counterfactuals found within guardrail constraints.",
            }
        export["results"].append(entry)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export, f, ensure_ascii=False, indent=4)
    print(f"✅ Results saved to {filename}")

save_results_to_json(cf_male, cf_female, cf_male_baseline, cf_female_baseline)


HE_DM_HbA1c value counts:
 HE_DM_HbA1c
-999.0    17438
 0.0      12810
 1.0       4392
Name: count, dtype: int64

HE_HP value counts:
 HE_HP
-999.0    16734
 0.0      11964
 1.0       5942
Name: count, dtype: int64
Risk class distribution:
integrated_target
0    6381
1    1370
2     649
3    1338
Name: count, dtype: int64
>>> Preprocessing complete. CSV saved.
df_final shape : (9738, 33)
Any string column: False

==================== Male Model Training ====================

[Male] Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.72      0.77       388
           1       0.32      0.33      0.32       120
           2       0.16      0.21      0.18        63
           3       0.49      0.57      0.53       143

    accuracy                           0.58       714
   macro avg       0.45      0.46      0.45       714
weighted avg       0.61      0.58      0.59       714


==================== Female Model Training ==========

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.27it/s]

✅ Found Normal pathways on attempt 1!
Query instance (original outcome : 3)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,26.398777,94.699997,73.800003,1298.084106,190.45079,19.077242,2359.64502,34.523525,9.989088,16.731873,2230.81665,54.438519,4.0,1.0,8.0,8.0,4.0,2.0,8.0,2.0,2.0,2.0,1.0,1.0,1.0,4.0,0.0,4.0,3.0,4.0,1.0,3



Diverse Counterfactual set (new outcome: 0)


...

No-guardrail DiCE result (check for hallucinations):
Query instance (original outcome : 3)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,40.145157,120.400002,96.199997,1005.9953,157.364014,13.051463,2699.26416,22.524097,6.350924,15.846289,2204.337891,40.448795,6.0,3.0,8.0,2.0,2.0,1.0,8.0,2.0,2.0,2.0,1.0,0.0,1.0,4.0,0.0,4.0,3.0,1.0,1.0,3



Diverse Counterfactual set (new outcome: 0)


,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,RiskClass
0,20.689863,68.5,49.9,-,169.3644,22.806234,-,14.270197,2.942819,22.934092,-,-,2.0,-,-,1.0,3.0,-,-,-,-,-,-,-,-,-,-,-,-,-,-,0.0
0,13.512497,53.2,44.0,129.8033,-,0.177327,0.7093,30.991793,0.106396,0.3525,31.2095,1.7118599,1.0,1.0,-,1.0,1.0,-,-,-,-,-,-,-,-,-,-,-,-,-,-,0.0
0,27.151768,86.4,74.1,1126.0991,193.7979,23.666834,-,16.951443,6.975961,21.361929,-,47.5705681,4.0,-,-,-,4.0,4.0,-,-,-,-,-,1.0,-,-,-,-,-,-,-,0.0
0,25.430452,77.8,58.6,-,193.1203,16.290781,-,19.164513,5.535898,19.785364,-,36.4128227,4.0,1.0,-,8.0,-,-,-,-,-,-,-,-,4.0,-,-,-,-,-,-,0.0


✅ Results saved to experiment_results.json
✅ Results saved to experiment_results.json
